# TFM Tenerife — Horarios y Calendario de Servicio (GTFS TITSA y Metropolitano)

Este notebook completa la ingesta GTFS iniciada en el notebook 7: convierte `stop_times.txt`,
`calendar.txt`, `calendar_dates.txt`, `trips.txt` y `routes.txt` en tablas relacionales,
preparando la base para el punto 4.5 (accesibilidad e isócronas).

## Paso 1 — Conectar con Neon

In [1]:
#pip install geopandas sqlalchemy psycopg2-binary geoalchemy2 shapely requests partridge python-dotenv


In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(r"C:\Users\mario\Desktop\TFM\.env")

db_user = os.environ['DB_USER']
db_password = os.environ['DB_PASSWORD']
db_host = os.environ['DB_HOST']
db_name = os.environ['DB_NAME']

conn_str = f"postgresql://{db_user}:{db_password}@{db_host}/{db_name}?sslmode=require"
engine = create_engine(conn_str, pool_pre_ping=True, pool_recycle=1800)

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. Version de PostGIS:', version)

Conectado. Version de PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## Paso 2 — Fuentes GTFS

Mismas fuentes que en el notebook 7 (TITSA y Metropolitano de Tenerife).

In [3]:
FUENTES_GTFS = {
    'titsa': {
        'url': 'https://datos.tenerife.es/ckan/dataset/36c2e26f-0d18-4b5a-b214-1636168e0765/resource/9f291323-8b78-453a-9008-4f0e3bfb3ce3/download/fichero-zip-de-google-transit.zip',
        'operador': 'TITSA',
        'modo': 'guagua',
    },
    'metropolitano': {
        'url': 'https://datos.tenerife.es/ckan/dataset/4b83e018-37d9-40a6-b6d1-1df2b91c8117/resource/7b7bbd1f-f53a-4413-a6b4-ea411e18c66d/download/fichero-zip-de-google-transit.zip',
        'operador': 'Metropolitano de Tenerife',
        'modo': 'tranvia',
    },
}

## Paso 3 — Descargar el GTFS a disco

partridge necesita el fichero guardado en disco. Con forzar=False no repite
la descarga si ya existe (útil mientras se prueba código); con forzar=True
siempre trae la versión más reciente publicada por el operador.

In [4]:
import os
import requests

def descargar_gtfs_a_disco(url, ruta_destino, forzar=False):
    if os.path.exists(ruta_destino) and not forzar:
        print(f'  {ruta_destino} ya existe, no se vuelve a descargar (forzar=False)')
        return ruta_destino
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with open(ruta_destino, 'wb') as f:
        f.write(resp.content)
    print(f'  {ruta_destino} descargado')
    return ruta_destino

## Paso 4 — Cargar el feed con partridge

Cargamos el feed completo (sin filtrar por día ni service_id), porque
modelamos el calendario completo con sus excepciones, no solo un día tipo.

In [5]:
import partridge as ptg

def cargar_feed(ruta_zip):
    return ptg.load_feed(ruta_zip)

## Paso 5 — Construir el calendario base (calendar.txt)

Qué días de la semana circula cada service_id, y entre qué fechas.

In [6]:
import pandas as pd

def construir_calendario(feed, operador):
    cal = feed.calendar.copy()
    if cal.empty:
        return pd.DataFrame(columns=[
            'service_id', 'operador', 'lunes', 'martes', 'miercoles', 'jueves',
            'viernes', 'sabado', 'domingo', 'fecha_inicio', 'fecha_fin'
        ])
    cal = cal.rename(columns={
        'monday': 'lunes', 'tuesday': 'martes', 'wednesday': 'miercoles',
        'thursday': 'jueves', 'friday': 'viernes', 'saturday': 'sabado', 'sunday': 'domingo',
        'start_date': 'fecha_inicio', 'end_date': 'fecha_fin'
    })
    cal['operador'] = operador
    cal['fecha_inicio'] = pd.to_datetime(cal['fecha_inicio'], format='%Y%m%d')
    cal['fecha_fin'] = pd.to_datetime(cal['fecha_fin'], format='%Y%m%d')
    cols = ['service_id', 'operador', 'lunes', 'martes', 'miercoles', 'jueves',
            'viernes', 'sabado', 'domingo', 'fecha_inicio', 'fecha_fin']
    return cal[cols]

## Paso 6 — Construir las excepciones de calendario (calendar_dates.txt)

Fechas concretas donde un service_id se añade (tipo 1) o se quita (tipo 2)
respecto al calendario base — festivos, servicios especiales, etc.

In [7]:
def construir_calendario_excepciones(feed, operador):
    exc = feed.calendar_dates.copy()
    if exc.empty:
        return pd.DataFrame(columns=['service_id', 'operador', 'fecha', 'tipo'])
    exc = exc.rename(columns={'date': 'fecha', 'exception_type': 'tipo'})
    exc['fecha'] = pd.to_datetime(exc['fecha'], format='%Y%m%d')
    exc['operador'] = operador
    return exc[['service_id', 'operador', 'fecha', 'tipo']]

## Paso 7 — Construir horarios (stop_times.txt)

Dos particularidades del formato GTFS:
- Las horas pueden superar 24:00:00 (servicios que cruzan la medianoche), por
  eso convertimos a segundos desde medianoche en vez de castear a TIME.
- Partridge ya convierte arrival_time/departure_time a segundos (float) al cargar el feed, no a strings "HH:MM:SS". _a_segundos contempla ambos casos por robustez ante otras librerías de carga.


In [12]:
def _a_segundos(valor):
    if pd.isna(valor):
        return None
    if isinstance(valor, str):
        h, m, s = map(int, valor.strip().split(':'))
        return h * 3600 + m * 60 + s
    return int(valor)  # ya viene en segundos desde partridge

def construir_horarios(feed, operador):
    st = feed.stop_times.copy()
    st['operador'] = operador
    st['arrival_seconds'] = st['arrival_time'].apply(_a_segundos)
    st['departure_seconds'] = st['departure_time'].apply(_a_segundos)
    cols = ['trip_id', 'stop_id', 'stop_sequence', 'operador',
            'arrival_seconds', 'departure_seconds']
    return st[[c for c in cols if c in st.columns]]

## Paso 8 — Construir viajes (trips.txt)

Conecta route_id (la línea) con service_id (cuándo circula) y con
stop_times a través de trip_id. TITSA no publica direction_id: no se puede
distinguir sentido ida/vuelta por esa columna para TITSA, usar
trip_headsign si se necesita esa distinción.

In [13]:
def construir_viajes(feed, operador):
    trips = feed.trips.copy()
    trips['operador'] = operador
    columnas_esperadas = ['trip_id', 'route_id', 'service_id', 'operador',
                          'trip_headsign', 'direction_id', 'shape_id']
    cols_presentes = [c for c in columnas_esperadas if c in trips.columns]
    return trips[cols_presentes]

## Paso 9 — Enriquecer rutas con atributos (routes.txt)

gtfs_rutas (notebook 7) solo tiene geometría. route_long_name de TITSA trae
entidades HTML sin escapar (&nbsp;) en algunos nombres; se limpia aquí.

In [14]:
import html

def construir_rutas_atributos(feed, operador):
    routes = feed.routes.copy()
    routes['operador'] = operador
    columnas_esperadas = ['route_id', 'route_short_name', 'route_long_name',
                          'route_type', 'operador', 'route_color']
    cols_presentes = [c for c in columnas_esperadas if c in routes.columns]
    routes = routes[cols_presentes]
    if 'route_long_name' in routes.columns:
        routes['route_long_name'] = routes['route_long_name'].apply(
            lambda x: html.unescape(x).strip() if isinstance(x, str) else x
        )
    return routes

## Paso 10 — Función para subir una tabla a Neon

Usa COPY (más rápido que to_sql fila a fila) — importante con
gtfs_horarios, que puede tener millones de filas.

In [15]:
import io

def subir_tabla(df, nombre, columnas_indice=None):
    df.head(0).to_sql(nombre, engine, schema='processed_data', if_exists='replace', index=False)

    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)

    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            cur.copy_expert(
                f"COPY processed_data.{nombre} FROM STDIN WITH (FORMAT csv)",
                buffer
            )
        raw_conn.commit()
    finally:
        raw_conn.close()

    if columnas_indice:
        with engine.begin() as conn:
            for col in columnas_indice:
                conn.execute(text(
                    f'CREATE INDEX IF NOT EXISTS idx_{nombre}_{col} '
                    f'ON processed_data.{nombre} ({col})'
                ))
    print('  ->', nombre, 'cargada (COPY):', len(df), 'filas')

## Paso 11 — Validar integridad

Comprueba que no hay trip_id/route_id "huérfanos" y que no hay duplicados
en las columnas que serán PRIMARY KEY. Se llama automáticamente antes de
crear las FK; si hay problemas, avisa mostrando conteos en vez de fallar
a ciegas al crear las constraints.

In [16]:
def validar_integridad():
    with engine.connect() as conn:
        horarios_huerfanos = pd.read_sql(text("""
            SELECT COUNT(*) FROM processed_data.gtfs_horarios h
            LEFT JOIN processed_data.gtfs_viajes v ON h.trip_id = v.trip_id
            WHERE v.trip_id IS NULL;
        """), conn).iloc[0, 0]

        viajes_huerfanos = pd.read_sql(text("""
            SELECT COUNT(*) FROM processed_data.gtfs_viajes v
            LEFT JOIN processed_data.gtfs_rutas_atributos r ON v.route_id = r.route_id
            WHERE r.route_id IS NULL;
        """), conn).iloc[0, 0]

        dup_viajes = pd.read_sql(text("""
            SELECT trip_id, COUNT(*) FROM processed_data.gtfs_viajes
            GROUP BY trip_id HAVING COUNT(*) > 1;
        """), conn)

        dup_rutas = pd.read_sql(text("""
            SELECT route_id, COUNT(*) FROM processed_data.gtfs_rutas_atributos
            GROUP BY route_id HAVING COUNT(*) > 1;
        """), conn)

    print('horarios sin viaje correspondiente:', horarios_huerfanos)
    print('viajes sin ruta correspondiente:', viajes_huerfanos)
    print('trip_id duplicados:', len(dup_viajes))
    print('route_id duplicados:', len(dup_rutas))

    hay_problemas = horarios_huerfanos > 0 or viajes_huerfanos > 0 or len(dup_viajes) > 0 or len(dup_rutas) > 0
    if hay_problemas:
        print('AVISO: hay inconsistencias, revisar antes de crear las FK')
    return not hay_problemas

## Paso 12 — Crear / eliminar PK y FK

Postgres exige que la columna referenciada por una FK tenga una restricción
de unicidad (PRIMARY KEY), no basta con un índice normal.

In [17]:
def crear_fks():
    with engine.begin() as conn:
        conn.execute(text("""
            ALTER TABLE processed_data.gtfs_rutas_atributos
            ADD CONSTRAINT pk_rutas_atributos PRIMARY KEY (route_id);
        """))
        conn.execute(text("""
            ALTER TABLE processed_data.gtfs_viajes
            ADD CONSTRAINT pk_viajes PRIMARY KEY (trip_id);
        """))
        conn.execute(text("""
            ALTER TABLE processed_data.gtfs_viajes
            ADD CONSTRAINT fk_viajes_rutas
            FOREIGN KEY (route_id) REFERENCES processed_data.gtfs_rutas_atributos(route_id);
        """))
        conn.execute(text("""
            ALTER TABLE processed_data.gtfs_horarios
            ADD CONSTRAINT fk_horarios_viajes
            FOREIGN KEY (trip_id) REFERENCES processed_data.gtfs_viajes(trip_id);
        """))
    print('PK y FK creadas')

def eliminar_fks():
    with engine.begin() as conn:
        conn.execute(text("""
            ALTER TABLE processed_data.gtfs_horarios
            DROP CONSTRAINT IF EXISTS fk_horarios_viajes;
        """))
        conn.execute(text("""
            ALTER TABLE processed_data.gtfs_viajes
            DROP CONSTRAINT IF EXISTS fk_viajes_rutas;
        """))
    print('FK eliminadas (si existían)')

## Paso 13 — Recarga completa del pipeline

Carga cada feed UNA sola vez por operador (antes se cargaba dos veces:
una para calendario/horarios, otra para viajes/rutas) y construye las 5
tablas de golpe. Elimina las FK antes de tocar las tablas y las recrea al
final, tras validar integridad.

In [18]:
import time

def recargar_todo_gtfs():
    eliminar_fks()
    calendarios_todos, excepciones_todas, horarios_todos = [], [], []
    viajes_todos, rutas_attr_todas = [], []

    for clave, info in FUENTES_GTFS.items():
        t0 = time.time()
        print('Procesando', info['operador'], '...')
        ruta_zip = descargar_gtfs_a_disco(info['url'], f'gtfs_{clave}.zip', forzar=True)
        feed = cargar_feed(ruta_zip)

        calendarios_todos.append(construir_calendario(feed, info['operador']))
        excepciones_todas.append(construir_calendario_excepciones(feed, info['operador']))
        horarios_todos.append(construir_horarios(feed, info['operador']))
        viajes_todos.append(construir_viajes(feed, info['operador']))
        rutas_attr_todas.append(construir_rutas_atributos(feed, info['operador']))
        print(f'  {info["operador"]}: {time.time()-t0:.1f}s')

    subir_tabla(pd.concat(rutas_attr_todas, ignore_index=True), 'gtfs_rutas_atributos', columnas_indice=['route_id'])
    subir_tabla(pd.concat(viajes_todos, ignore_index=True), 'gtfs_viajes', columnas_indice=['trip_id', 'route_id', 'service_id'])
    subir_tabla(pd.concat(calendarios_todos, ignore_index=True), 'gtfs_calendario', columnas_indice=['service_id'])
    subir_tabla(pd.concat(excepciones_todas, ignore_index=True), 'gtfs_calendario_excepciones', columnas_indice=['service_id', 'fecha'])
    subir_tabla(pd.concat(horarios_todos, ignore_index=True), 'gtfs_horarios', columnas_indice=['trip_id', 'stop_id'])

    if validar_integridad():
        crear_fks()
    else:
        print('FK NO creadas por inconsistencias — revisar antes de reintentar')

    print('Recarga completa terminada')

## Paso 14 — Ejecutar

Único punto de ejecución real del notebook.

In [19]:
recargar_todo_gtfs()

FK eliminadas (si existían)
Procesando TITSA ...
  gtfs_titsa.zip descargado
  TITSA: 35.9s
Procesando Metropolitano de Tenerife ...
  gtfs_metropolitano.zip descargado
  Metropolitano de Tenerife: 0.9s
  -> gtfs_rutas_atributos cargada (COPY): 180 filas
  -> gtfs_viajes cargada (COPY): 74567 filas
  -> gtfs_calendario cargada (COPY): 3 filas
  -> gtfs_calendario_excepciones cargada (COPY): 28081 filas
  -> gtfs_horarios cargada (COPY): 2082154 filas
horarios sin viaje correspondiente: 0
viajes sin ruta correspondiente: 0
trip_id duplicados: 0
route_id duplicados: 0
PK y FK creadas
Recarga completa terminada


## Paso 15 — Verificar resultados

In [20]:
with engine.connect() as conn:
    resumen_calendario = pd.read_sql(
        text('SELECT operador, COUNT(*) FROM processed_data.gtfs_calendario GROUP BY operador;'), conn
    )
    resumen_horarios = pd.read_sql(
        text('SELECT operador, COUNT(*) FROM processed_data.gtfs_horarios GROUP BY operador;'), conn
    )

display(resumen_calendario)
display(resumen_horarios)

,operador,count
0,Metropolitano de Tenerife,3


,operador,count
0,Metropolitano de Tenerife,162
1,TITSA,2081992


In [ ]:
SELECT COUNT(*) FILTER (WHERE arrival_seconds IS NULL) AS nulos_arrival,
    COUNT(*) FILTER (WHERE departure_seconds IS NULL) AS nulos_departure,
    COUNT(*) AS total
FROM processed_data.gtfs_horarios;   COUNT(*) FILTER (WHERE departure_seconds IS NULL) AS nulos_departure,
       COUNT(*) AS total
FROM processed_data.gtfs_horarios;

SyntaxError: Invalid star expression (1267501906.py, line 1)

## Notas

- gtfs_horarios puede tener millones de filas; se usa COPY en vez de to_sql
  fila a fila para que la carga sea rápida (ver subir_tabla).

- Cada feed GTFS se carga UNA sola vez por operador dentro de
  recargar_todo_gtfs() (antes se cargaba dos veces: una para
  calendario/horarios, otra para viajes/rutas — optimización aplicada).

- Algunas filas de gtfs_horarios tienen arrival_seconds/departure_seconds a
  NULL (paradas interpoladas sin horario explícito en el GTFS). Pendiente
  resolver antes del 4.5: interpolar estos valores o filtrarlos según lo
  que necesite el cálculo de isócronas.

- TITSA no usa calendar.txt (0 filas): define todo su servicio mediante
  calendar_dates.txt (~28.000 filas de excepciones puntuales por fecha).
  Metropolitano de Tenerife sí usa calendar.txt con patrón semanal
  recurrente. Para saber si un service_id de TITSA circula en una fecha
  dada, consultar SIEMPRE gtfs_calendario_excepciones, no gtfs_calendario.

- gtfs_viajes: TITSA no publica direction_id; usar trip_headsign si se
  necesita distinguir sentido ida/vuelta para TITSA.

- gtfs_rutas_atributos y gtfs_viajes tienen PRIMARY KEY (route_id / trip_id),
  necesarias para que las FK funcionen.

- Para recargar el pipeline completo, usar ÚNICAMENTE recargar_todo_gtfs()
  (Paso 14) — no ejecutar subir_tabla() suelto sobre tablas con FK activas.

- Pendiente para el 4.5: usar gtfs_horarios + gtfs_calendario/
  gtfs_calendario_excepciones para calcular tiempos de viaje reales entre
  paradas, según el operador.